# Module 3: Data Cleaning
**Real-Time Payment Fraud Detection for an E-Commerce Platform**

All cleaning decisions below are directly grounded in the empirical findings of the EDA (`EDA-E-commerce_Fraud_Detection.ipynb` & `Business-Understanding-EDA-14-10.pdf`):
1. **Completeness, Duplicates & Category Consistency (`[Q1.1]`)**: Verify `0` null cells, `0` duplicate rows, `0` negative amounts/balances, valid sender prefix `C`, and destination prefixes `C`/`M`. Ensure monotonic `step` order.
2. **Scope Filtering (`[Q2.2]`)**: All `8,213` fraud cases occur strictly in `TRANSFER` (`4,097` cases; `0.77%` rate) and `CASH_OUT` (`4,116` cases; `0.18%` rate). `CASH_IN`, `DEBIT`, and `PAYMENT` contain `0` fraud cases. Modeling is restricted strictly to `TRANSFER` and `CASH_OUT` (`2,770,409` rows; `100%` of fraud).
3. **Outliers & Simulator Artefacts Treatment (`[Q1.3]`)**:
   * **High-value outliers ($> Q_3 + 3 \times IQR$)**: In `CASH_OUT` and `TRANSFER`, fraud rates inside outliers (`43.56%` and `2.84%`) are far higher than outside (`0.11%` and `0.69%`). Outliers are risk indicators and **must not be deleted**.
   * **Zero-amount transactions (`amount == 0`)**: Exactly `16` rows in the log have `amount == 0`, and all `16` are labeled `isFraud == 1` (simulator artefact). Flagged explicitly via `is_zero_amount` rather than dropped.
   * **10M simulator cap (`amount == 10,000,000`)**: Fraudulent amounts are capped at `10,000,000` (`287` fraud rows vs `2,920` legitimate rows at `10M`, and `2,443` legitimate rows above `10M`).
4. **Leakage Exclusion (`[Q1.4]`)**: Dataset authors state cancelled fraud transactions leave balances inconsistent (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`). These four balance columns and the baseline rule column (`isFlaggedFraud`) are strictly excluded from predictive features.

In [1]:
# ===== [M3.0] Setup & Load Augmented Dataset from Module 1 =====
import os, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
np.random.seed(42)

DATA_PATH = r"C:\LONG\Project\BA Project\PS_20174392719_1491204439457_log.csv"
OUT_DIR = r"C:\LONG\Project\BA Project\outputs"
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
AUGMENTED_PATH = f"{OUT_DIR}/paysim_augmented_module1.parquet"

TYPES = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
MODEL_TYPES = ["TRANSFER", "CASH_OUT"]
LEAKAGE_COLS = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
RULE_COL = ["isFlaggedFraud"]

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

if os.path.exists(AUGMENTED_PATH):
    df_raw = pd.read_parquet(AUGMENTED_PATH)
else:
    # Load raw PaySim and run the exact Module 1 synthetic generator
    df_raw = pd.read_csv(DATA_PATH)
    df_raw["type"] = df_raw["type"].astype(str).str.strip().str.upper().str.replace("-", "_", regex=False)
    if not df_raw["step"].is_monotonic_increasing:
        df_raw = df_raw.sort_values("step", kind="stable").reset_index(drop=True)
    rng = np.random.default_rng(42)
    n_rows = len(df_raw)
    is_fraud = df_raw["isFraud"].to_numpy().astype(bool)
    n_fraud, n_legit = int(is_fraud.sum()), n_rows - int(is_fraud.sum())
    
    age_legit = rng.gamma(3.0, 150.0, size=n_legit)
    young_mask = rng.random(n_fraud) < 0.70
    age_fraud = np.empty(n_fraud)
    age_fraud[young_mask] = rng.exponential(25.0, size=int(young_mask.sum()))
    age_fraud[~young_mask] = rng.gamma(3.0, 150.0, size=int((~young_mask).sum()))
    acc_age = np.empty(n_rows, dtype=np.int32)
    acc_age[~is_fraud] = np.clip(np.rint(age_legit), 1, 3650)
    acc_age[is_fraud] = np.clip(np.rint(age_fraud), 1, 3650)
    df_raw["account_age_days"] = acc_age
    
    trust = np.empty(n_rows, dtype=np.float32)
    trust[~is_fraud] = rng.beta(8.0, 2.0, size=n_legit)
    trust[is_fraud] = rng.beta(2.0, 6.0, size=n_fraud)
    df_raw["device_trust_score"] = np.clip(trust, 0.0, 1.0)
    
    new_dev = np.empty(n_rows, dtype=np.int8)
    new_dev[~is_fraud] = (rng.random(n_legit) < 0.08).astype(np.int8)
    new_dev[is_fraud] = (rng.random(n_fraud) < 0.72).astype(np.int8)
    df_raw["is_new_device"] = new_dev
    
    dist_legit = rng.lognormal(2.5, 1.2, size=n_legit)
    out_m = rng.random(n_legit) < 0.03
    dist_legit[out_m] = rng.uniform(500.0, 5000.0, size=int(out_m.sum()))
    dist_fraud = rng.lognormal(6.2, 1.5, size=n_fraud)
    dist = np.empty(n_rows, dtype=np.float32)
    dist[~is_fraud] = np.clip(dist_legit, 0.0, 15000.0)
    dist[is_fraud] = np.clip(dist_fraud, 0.0, 15000.0)
    df_raw["ip_billing_distance_km"] = dist
    
    mismatch = np.empty(n_rows, dtype=np.int8)
    mismatch[~is_fraud] = (rng.random(n_legit) < 0.05).astype(np.int8)
    mismatch[is_fraud] = (rng.random(n_fraud) < 0.60).astype(np.int8)
    df_raw["shipping_billing_mismatch"] = mismatch
    
    auth = np.empty(n_rows, dtype=np.int8)
    auth[~is_fraud] = np.clip(rng.poisson(0.15, size=n_legit), 0, 15)
    auth[is_fraud] = np.clip(rng.poisson(2.80, size=n_fraud), 0, 15)
    df_raw["failed_auth_attempts_24h"] = auth

In [2]:
# ===== [M3.1] Step-by-Step Cleaning Pipeline with Before/After Documentation =====
def audit_snapshot(d: pd.DataFrame, stage: str) -> dict:
    return {
        "stage": stage,
        "rows": len(d),
        "columns": d.shape[1],
        "null_cells": int(d.isna().sum().sum()),
        "duplicate_rows": int(d.duplicated().sum()),
        "fraud_cases": int(d["isFraud"].sum()),
        "fraud_rate_pct": float(100 * d["isFraud"].mean()),
        "total_payment_value": float(d["amount"].sum()),
        "fraud_value": float((d["amount"] * d["isFraud"]).sum()),
        "memory_mb": float(d.memory_usage(deep=True).sum() / (1024 ** 2))
    }

cleaning_log = [audit_snapshot(df_raw, "1. Before Cleaning (All transaction types + Balance cols)")]

# Step 1: Verify string standardization, duplicates, and missing values
df_raw["type"] = df_raw["type"].astype(str).str.strip().str.upper().str.replace("-", "_", regex=False)
df_raw["nameOrig"] = df_raw["nameOrig"].astype(str).str.strip()
df_raw["nameDest"] = df_raw["nameDest"].astype(str).str.strip()
assert df_raw["type"].isin(TYPES).all(), "Invalid transaction type found"
assert int(df_raw.isna().sum().sum()) == 0, "Unexpected null values found"
assert int(df_raw.duplicated().sum()) == 0, "Unexpected duplicate rows found"

if not df_raw["step"].is_monotonic_increasing:
    df_raw = df_raw.sort_values("step", kind="stable").reset_index(drop=True)

# Step 2: Compute 3xIQR outlier fences per transaction type (matching [Q1.3])
g_amt = df_raw.groupby("type", observed=True)["amount"]
q1, q3 = g_amt.quantile(0.25), g_amt.quantile(0.75)
upper_fence_3xIQR = (q3 + 3.0 * (q3 - q1)).to_dict()
df_raw["is_amount_outlier_3xIQR"] = (df_raw["amount"] > df_raw["type"].map(upper_fence_3xIQR)).astype("int8")
df_raw["is_zero_amount"] = (df_raw["amount"] == 0).astype("int8")

# Step 3: Restrict scope strictly to TRANSFER and CASH_OUT (matching [Q2.2] Takeaway)
df_clean = df_raw[df_raw["type"].isin(MODEL_TYPES)].copy().reset_index(drop=True)
cleaning_log.append(audit_snapshot(df_clean, "2. After Scope Restriction (TRANSFER + CASH_OUT only)"))

# Step 4: Drop post-decision balance leakage columns & separate baseline rule column (matching [Q1.4])
df_clean.drop(columns=[c for c in (LEAKAGE_COLS + RULE_COL) if c in df_clean.columns], inplace=True)
cleaning_log.append(audit_snapshot(df_clean, "3. After Removing Balance Leakage & Baseline Rule Columns"))

# Step 5: Optimize data types
df_clean["step"] = df_clean["step"].astype("int32")
df_clean["type"] = pd.Categorical(df_clean["type"], categories=MODEL_TYPES)
df_clean["isFraud"] = df_clean["isFraud"].astype("int8")
cleaning_log.append(audit_snapshot(df_clean, "4. Final Cleaned Dataset (Optimized Dtypes)"))

before_after_df = pd.DataFrame(cleaning_log).set_index("stage")
before_after_df.to_csv(f"{OUT_DIR}/tables/M3_1_before_after_cleaning_comparison.csv")
display(before_after_df)

# Save cleaned dataset for Module 4
clean_path = f"{OUT_DIR}/paysim_cleaned_module3.parquet"
df_clean.to_parquet(clean_path, index=False)
print(f"Saved cleaned dataset to: {clean_path}")

,rows,columns,null_cells,duplicate_rows,fraud_cases,fraud_rate_pct,total_payment_value,fraud_value,memory_mb
stage,,,,,,,,,
1. Before Cleaning (All transaction types + Balance cols),6362620,17,0,0,8213,0.13,"1,144,392,944,759.77","12,056,415,427.84",797.24
2. After Scope Restriction (TRANSFER + CASH_OUT only),2770409,19,0,0,8213,0.30,"879,704,982,487.66","12,056,415,427.84",354.93
3. After Removing Balance Leakage & Baseline Rule Columns,2770409,14,0,0,8213,0.30,"879,704,982,487.66","12,056,415,427.84",249.25
4. Final Cleaned Dataset (Optimized Dtypes),2770409,14,0,0,8213,0.30,"879,704,982,487.66","12,056,415,427.84",180.23


Saved cleaned dataset to: C:\LONG\Project\BA Project\outputs/paysim_cleaned_module3.parquet
